This Notebook is used to conduct the graph analysis for the paper.

Here are contents:
1. whether the standard deviation of three replicate for local fitness within 10%
2. check joyplot:dead mutant and wildtype in 0 concentration

In [ ]:
# imports
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import colors

from repro_helpers import preprocess_data
from repro_helpers import calculate_normalized_fitness

# Loading Project Data

In [ ]:
import os
from repro_helpers import REPO_ROOT, REPO_DATA

# Portable `base_path`. The notebook writes any
# figures under ./_generated (gitignored); all data is read from the repo via
# repro_helpers, which reimplements src.preprocess / src.pair_table_global.
base_path = str(REPO_ROOT / "src" / "graph_analysis" / "_generated")
os.makedirs(f"{base_path}/data/outputs/figures/summary-stats", exist_ok=True)

wildtype = "............."
dead_mutant = "XXXXXXXXXXXXX"

# File paths
amp_path = f"{base_path}/data/raw/combined-auc/genotype_auc_sorted_ampicillin.csv"
azt_path = f"{base_path}/data/raw/combined-auc/genotype_auc_sorted_aztreonam.csv"
amp_pairs_path = f"{base_path}/data/processed/amp_pairs.csv"
azt_pairs_path = f"{base_path}/data/processed/azt_pairs.csv"

# Load and preprocess data
processed_data = preprocess_data(amp_path, azt_path, amp_pairs_path, azt_pairs_path, clean_nulls_flag=True)

# Access the processed dataframes
amp_df = processed_data['amp']['original']
amp_long_df = processed_data['amp']['long']
amp_pairs_df = processed_data['amp']['pairs']

azt_df = processed_data['azt']['original']
azt_long_df = processed_data['azt']['long']
azt_pairs_df = processed_data['azt']['pairs']


In [ ]:
def split_mutant(mutant):
    mid = len(mutant) // 2
    return mutant[:mid], mutant[mid:]

# def calculate_statistics(df):
#     rep1 = df['replicate1'].to_numpy()
#     rep2 = df['replicate2'].to_numpy()
#     rep3 = df['replicate3'].to_numpy()
#     rep_arr = np.stack([rep1, rep2, rep3], axis=1)

#     # convert back to exponential scale
#     rep_arr = np.power(10, rep_arr)

#     mean = np.nanmean(rep_arr, axis=1)
#     median = np.nanmedian(rep_arr, axis=1)
#     std = np.nanstd(rep_arr, axis=1)
    
#     # Handle division by zero for cv_std
#     cv_std = np.zeros_like(std)
#     nonzero_mean = mean != 0
#     cv_std[nonzero_mean] = std[nonzero_mean] / mean[nonzero_mean]
    
#     # Handle division by zero for z_scores
#     z_scores = np.zeros_like(rep_arr)
#     nonzero_std = std != 0
#     z_scores[nonzero_std] = (rep_arr[nonzero_std] - mean[nonzero_std, None]) / std[nonzero_std, None]
    
#     z_median = np.nanmedian(z_scores, axis=1)

#     mutant_profiles = df['mutant_profile'].to_numpy()
#     mutant_part1, mutant_part2 = zip(*[split_mutant(mutant) for mutant in mutant_profiles])

#     stats_df = pl.DataFrame({
#         'mutant_profile': df['mutant_profile'],
#         'concentration': df['concentration'],
#         'mutant_part1': mutant_part1,
#         'mutant_part2': mutant_part2,
#         'replicate1': df['replicate1'],
#         'replicate2': df['replicate2'],
#         'replicate3': df['replicate3'],
#         'mean': np.log10(mean),
#         'median': np.log10(median),
#         'std': np.log10(std),
#         'cv_std': np.log10(cv_std),
#         'z_median': np.log10(z_median),
#         'z_scores1': z_scores[:, 0],
#         'z_scores2': z_scores[:, 1],
#         'z_scores3': z_scores[:, 2]
#     })

#     return stats_df

def calculate_statistics(df):
    rep1 = df['replicate1'].to_numpy()
    rep2 = df['replicate2'].to_numpy()
    rep3 = df['replicate3'].to_numpy()
    rep_arr = np.stack([rep1, rep2, rep3], axis=1)

    # Calculate statistics directly in log10 space
    mean = np.nanmean(rep_arr, axis=1)
    median = np.nanmedian(rep_arr, axis=1)
    std = np.nanstd(rep_arr, axis=1)
    
    # Handle division by zero for cv_std
    cv_std = np.zeros_like(std)
    nonzero_mean = mean != 0
    cv_std[nonzero_mean] = std[nonzero_mean] / mean[nonzero_mean]
    
    # Handle division by zero for z_scores
    z_scores = np.zeros_like(rep_arr)
    nonzero_std = std != 0
    z_scores[nonzero_std] = (rep_arr[nonzero_std] - mean[nonzero_std, None]) / std[nonzero_std, None]
    
    z_median = np.nanmedian(z_scores, axis=1)

    mutant_profiles = df['mutant_profile'].to_numpy()
    mutant_part1, mutant_part2 = zip(*[split_mutant(mutant) for mutant in mutant_profiles])

    stats_df = pl.DataFrame({
        'mutant_profile': df['mutant_profile'],
        'concentration': df['concentration'],
        'mutant_part1': mutant_part1,
        'mutant_part2': mutant_part2,
        'replicate1': df['replicate1'],
        'replicate2': df['replicate2'],
        'replicate3': df['replicate3'],
        'mean': mean,
        'median': median,
        'std': std,
        'cv_std': cv_std,
        'z_median': z_median,
        'z_scores1': z_scores[:, 0],
        'z_scores2': z_scores[:, 1],
        'z_scores3': z_scores[:, 2]
    })

    return stats_df

def substract_wildtype(df, wildtype="............."):
    # Filter wildtype data
    wildtype_df = df.filter(pl.col("mutant_profile") == wildtype)

    # Create dictionaries for wildtype mean and median values
    wildtype_mean_dict = dict(zip(wildtype_df["concentration"], wildtype_df["mean"]))
    wildtype_median_dict = dict(zip(wildtype_df["concentration"], wildtype_df["median"]))

    # Create new columns with explicit return dtype
    df = df.with_columns([
        (pl.col("mean") - pl.col("concentration").map_elements(
            lambda x: wildtype_mean_dict.get(x), 
            return_dtype=pl.Float64
        )).alias("mean_w"),
        (pl.col("median") - pl.col("concentration").map_elements(
            lambda x: wildtype_median_dict.get(x),
            return_dtype=pl.Float64
        )).alias("median_w")
    ])

    return df

In [ ]:
amp_stats_df = calculate_statistics(amp_long_df)
azt_stats_df = calculate_statistics(azt_long_df)

amp_stats_df = substract_wildtype(amp_stats_df)
azt_stats_df = substract_wildtype(azt_stats_df)

In [ ]:
amp_stats_df = amp_stats_df.sort("mutant_profile")
azt_stats_df = azt_stats_df.sort("mutant_profile")

In [ ]:
amp_global_fitness_df = calculate_normalized_fitness(amp_long_df)
azt_global_fitness_df = calculate_normalized_fitness(azt_long_df)

In [ ]:
# Add mutant parts columns to global fitness dataframes
amp_global_fitness_df = amp_global_fitness_df.with_columns([
    pl.col('mutant_profile').map_elements(lambda x: split_mutant(x)[0], return_dtype=str).alias('mutant_part1'),
    pl.col('mutant_profile').map_elements(lambda x: split_mutant(x)[1], return_dtype=str).alias('mutant_part2')
])
azt_global_fitness_df = azt_global_fitness_df.with_columns([
    pl.col('mutant_profile').map_elements(lambda x: split_mutant(x)[0], return_dtype=str).alias('mutant_part1'),
    pl.col('mutant_profile').map_elements(lambda x: split_mutant(x)[1], return_dtype=str).alias('mutant_part2')
])


In [ ]:
amp_global_fitness_df = amp_global_fitness_df.sort("mutant_profile")
azt_global_fitness_df = azt_global_fitness_df.sort("mutant_profile")

In [ ]:
from repro_helpers import load_global_epistasis

# Per-drug global epistasis tables, sliced from data/processed/Epistasis_Combined.parquet
# at the representative concentration (AMP 781 / AZT 36), matching 05_epistasis_figures.ipynb.
epistasis_df, amp_global_epistasis_df, azt_global_epistasis_df = load_global_epistasis()

# Analysis

## Check whether the standard deviation of three replicates for local fitness is below 10%. Draw a scatter correlation

For each mutant, at each concentration and for each drug, the mean and standard deviation across the three replicates are computed and plotted as a mean-versus-SD scatter. `calculate_statistics` computes these in linear space; expressed on a log10 scale, a 10% coefficient of variation corresponds to `std < mean - 1`. Under this criterion the majority of mutants fall within the 10% coefficient-of-variation window (see the results below).

In [ ]:
amp_long_df

In [ ]:
amp_stats_df

In [ ]:
np.std([3.288635,3.15538, 3.405358	 ])

In [ ]:
def summarize_cv_below_10_percent_linear(stats_df, label=""):
    # Ignore rows where mean or std is not finite
    valid = stats_df.filter(pl.col("mean").is_finite() & pl.col("std").is_finite())

    passing = valid.filter(pl.col("std") < (0.1 * pl.col("mean")))
    total = valid.height
    passed = passing.height
    print(f"{label}: {passed}/{total} ({passed/total:.1%}) rows have SD < 10% of mean (linear space).")

def summarize_cv_by_concentration_linear(stats_df, label=""):
    concentrations = stats_df["concentration"].unique().to_list()
    print(f"\nSummary for {label} (linear space):")
    print(f"{'Concentration':>15} | {'Pass':>6} | {'Total':>6} | {'% Pass':>7}")
    print("-" * 45)
    for conc in concentrations:
        df_conc = stats_df.filter(pl.col("concentration") == conc)
        valid = df_conc.filter(pl.col("mean").is_finite() & pl.col("std").is_finite())

        passing = valid.filter(pl.col("std") < (0.1 * pl.col("mean")))
        total = valid.height
        passed = passing.height
        percent = (passed / total * 100) if total > 0 else 0
        print(f"{str(conc):>15} | {passed:6} | {total:6} | {percent:7.1f}")

# Usage:
summarize_cv_below_10_percent_linear(amp_stats_df, label="Ampicillin")
summarize_cv_below_10_percent_linear(azt_stats_df, label="Aztreonam")

summarize_cv_by_concentration_linear(amp_stats_df, label="Ampicillin")
summarize_cv_by_concentration_linear(azt_stats_df, label="Aztreonam")

In [ ]:


def hist2d_cv_10pct(stats_df, label=""):
    # Font settings following Nature journal guidelines
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['DejaVu Sans']  # Default to a reliable system font
    
    # Nature-compliant figure styling
    plt.rcParams['font.size'] = 12        # Base font size
    plt.rcParams['axes.titlesize'] = 12   # Title size
    plt.rcParams['axes.labelsize'] = 12   # Axis label size
    plt.rcParams['xtick.labelsize'] = 10  # Tick label size
    plt.rcParams['ytick.labelsize'] = 10  # Tick label size

    # Filter valid rows
    valid = stats_df.filter(pl.col("mean").is_finite() & pl.col("std").is_finite())
    mean = valid["mean"].to_numpy()
    std = valid["std"].to_numpy()

    plt.figure(figsize=(8, 6))
    # 2D histogram with viridis colormap and log color scale
    h = plt.hist2d(mean, std, bins=500, cmap='viridis', norm=plt.matplotlib.colors.LogNorm())
    # h = plt.hist2d(mean, std, bins=300, cmap='viridis')
    plt.colorbar(label='Count (log scale)')
    # Plot the threshold line: y = 0.1x
    x_vals = np.linspace(mean.min(), mean.max(), 100)
    plt.plot(x_vals, 0.1 * x_vals, 'r--', label='10% threshold (std = 0.1 × mean)')
    plt.xlabel('Mean')
    plt.ylabel('Std')
    plt.title(f'Standard Deviation vs Mean for {label}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/sd-vs-mean-10%-{label}.png", bbox_inches="tight", dpi=1000, format="png")
    plt.show()

# Usage:
hist2d_cv_10pct(amp_stats_df, label="Ampicillin")
hist2d_cv_10pct(azt_stats_df, label="Aztreonam")

## Check joyplot, wildtype and dead mutant in concentration 0
dead mutant blue dash line is not visible at concentration 0, because the two mutants have very close fitness values.

In [ ]:
dead_mutant

In [ ]:
amp_long_df.filter(pl.col("mutant_profile") == dead_mutant).filter(pl.col("concentration") == 0)

In [ ]:
amp_long_df.filter(pl.col("mutant_profile") == wildtype).filter(pl.col("concentration") == 0)

In [ ]:
azt_long_df.filter(pl.col("mutant_profile") == dead_mutant).filter(pl.col("concentration") == 0)


In [ ]:
azt_long_df.filter(pl.col("mutant_profile") == wildtype).filter(pl.col("concentration") == 0)